# DarkSpot — парсинг конкурентов (2GIS)

Собираем конкурентов по всем районам Москвы, считаем коэффициент конкурентности, сохраняем в CSV.

In [1]:
!pip install requests pandas numpy folium tqdm -q

In [2]:
import requests
import pandas as pd
import numpy as np
import folium
import time
from tqdm import tqdm
from dataclasses import dataclass
from typing import Optional


## 1. Конфиг

In [14]:
TWOGIS_API_KEY = "107f1407-46a1-44ba-a459-72018232f9e2"
BASE_URL       = "https://catalog.api.2gis.com/3.0/items"

BUSINESS_TYPE    = "coffeeshop"
TARGET_AUDIENCE  = ["students", "office_workers"]
SEARCH_RADIUS_M  = 800


## 2. Маппинги и районы

In [15]:
@dataclass
class ClientInput:
    business_type:    str
    target_audience:  list
    location_name:    str
    lat:              float
    lon:              float
    search_radius_m:  int = 1000


BUSINESS_TYPE_MAP = {
    "coffeeshop": {
        "queries":           ["кофейня", "кофе", "кафе"],
        "rubrics":           ["cafe", "coffee"],
        "direct_keywords":   ["кофе", "coffee", "espresso", "specialty", "капучино"],
        "indirect_keywords": ["чай", "выпечка", "десерт", "булочная", "кондитерская"],
    },
    "nail_salon": {
        "queries":           ["маникюр", "ногтевой сервис", "nail"],
        "rubrics":           ["beauty", "nail"],
        "direct_keywords":   ["маникюр", "педикюр", "ногти", "nail", "гель"],
        "indirect_keywords": ["салон красоты", "spa", "брови", "ресницы"],
    },
    "barbershop": {
        "queries":           ["барбершоп", "парикмахерская", "мужская стрижка"],
        "rubrics":           ["barbershop", "hair"],
        "direct_keywords":   ["барбер", "barbershop", "мужская стрижка", "борода"],
        "indirect_keywords": ["парикмахерская", "салон красоты", "стрижка"],
    },
    "restaurant": {
        "queries":           ["ресторан"],
        "rubrics":           ["restaurant"],
        "direct_keywords":   ["ресторан", "restaurant"],
        "indirect_keywords": ["кафе", "бистро", "столовая", "суши", "пицца"],
    },
}

TARGET_AUDIENCE_WEIGHTS = {
    "students":       {"price_sensitivity": 0.8, "preferred_formats": ["кофейня", "fast food", "столовая"]},
    "office_workers": {"price_sensitivity": 0.4, "preferred_formats": ["кофейня", "бизнес-ланч", "ресторан"]},
    "families":       {"price_sensitivity": 0.5, "preferred_formats": ["кафе", "ресторан", "пиццерия"]},
    "tourists":       {"price_sensitivity": 0.3, "preferred_formats": ["ресторан", "кафе", "кофейня"]},
    "young_adults":   {"price_sensitivity": 0.6, "preferred_formats": ["кофейня", "бар", "фастфуд"]},
}

# центроиды районов Москвы (name, okrug, lat, lon)
MOSCOW_DISTRICTS = [
    ("Хамовники","ЦАО",55.728,37.58),("Арбат","ЦАО",55.749,37.592),("Тверской","ЦАО",55.767,37.606),
    ("Пресненский","ЦАО",55.76,37.57),("Замоскворечье","ЦАО",55.735,37.63),("Якиманка","ЦАО",55.73,37.61),
    ("Басманный","ЦАО",55.765,37.67),("Таганский","ЦАО",55.74,37.66),("Мещанский","ЦАО",55.778,37.633),
    ("Красносельский","ЦАО",55.78,37.66),("Аэропорт","САО",55.8,37.53),("Сокол","САО",55.805,37.515),
    ("Беговой","САО",55.785,37.56),("Тимирязевский","САО",55.82,37.57),("Савёловский","САО",55.795,37.585),
    ("Войковский","САО",55.82,37.5),("Коптево","САО",55.84,37.52),("Головинский","САО",55.85,37.49),
    ("Хорошёвский","САО",55.78,37.53),("Останкинский","СВАО",55.82,37.61),
    ("Бабушкинский","СВАО",55.87,37.66),("Свиблово","СВАО",55.86,37.63),("Алексеевский","СВАО",55.81,37.64),
    ("Марьина роща","СВАО",55.8,37.615),("Бутырский","СВАО",55.815,37.585),("Ростокино","СВАО",55.835,37.66),
    ("Отрадное","СВАО",55.865,37.605),("Бибирево","СВАО",55.89,37.605),("Сокольники","ВАО",55.79,37.68),
    ("Измайлово","ВАО",55.79,37.78),("Преображенское","ВАО",55.795,37.715),("Богородское","ВАО",55.815,37.71),
    ("Перово","ВАО",55.75,37.78),("Новогиреево","ВАО",55.75,37.81),("Вешняки","ВАО",55.72,37.82),
    ("Люблино","ЮВАО",55.68,37.76),("Марьино","ЮВАО",55.65,37.74),("Текстильщики","ЮВАО",55.705,37.73),
    ("Кузьминки","ЮВАО",55.7,37.77),("Рязанский","ЮВАО",55.72,37.785),("Нижегородский","ЮВАО",55.73,37.715),
    ("Лефортово","ЮВАО",55.76,37.7),("Даниловский","ЮАО",55.71,37.63),("Донской","ЮАО",55.705,37.61),
    ("Чертаново Центральное","ЮАО",55.63,37.61),("Чертаново Северное","ЮАО",55.65,37.605),
    ("Нагорный","ЮАО",55.67,37.62),("Царицыно","ЮАО",55.62,37.68),
    ("Орехово-Борисово Северное","ЮАО",55.62,37.73),("Гагаринский","ЮЗАО",55.7,37.57),
    ("Академический","ЮЗАО",55.69,37.57),("Обручевский","ЮЗАО",55.66,37.54),
    ("Коньково","ЮЗАО",55.63,37.52),("Ломоносовский","ЮЗАО",55.69,37.54),
    ("Черёмушки","ЮЗАО",55.67,37.56),("Ясенево","ЮЗАО",55.6,37.53),
    ("Раменки","ЗАО",55.7,37.5),("Дорогомилово","ЗАО",55.74,37.55),("Кунцево","ЗАО",55.73,37.43),
    ("Крылатское","ЗАО",55.76,37.41),("Фили-Давыдково","ЗАО",55.73,37.47),("Можайский","ЗАО",55.725,37.41),
    ("Очаково-Матвеевское","ЗАО",55.69,37.46),("Тропарёво-Никулино","ЗАО",55.66,37.48),
    ("Строгино","СЗАО",55.8,37.4),("Хорошёво-Мнёвники","СЗАО",55.78,37.47),("Щукино","СЗАО",55.81,37.46),
    ("Северное Тушино","СЗАО",55.855,37.44),("Южное Тушино","СЗАО",55.83,37.43),("Митино","СЗАО",55.84,37.36),
]


## 3. Оригинальные функции парсинга и скоринга

In [16]:
def parse_place(raw: dict) -> dict:
    rubrics = raw.get("rubrics", [])
    rubric_names = [r.get("name", "") for r in rubrics]
    point = raw.get("point", {})
    reviews = raw.get("reviews", {})
    rating = reviews.get("general_rating", None)
    review_count = reviews.get("general_review_count", 0)
    return {
        "id":           raw.get("id"),
        "name":         raw.get("name", ""),
        "rubrics":      ", ".join(rubric_names),
        "address":      raw.get("address_name", ""),
        "lat":          point.get("lat"),
        "lon":          point.get("lon"),
        "rating":       float(rating) if rating else None,
        "review_count": int(review_count),
    }


def fetch_2gis_places(query: str, lat: float, lon: float, radius: int, api_key: str) -> list:
    results = []
    page = 1
    while True:
        params = {
            "q": query, "point": f"{lon},{lat}", "radius": radius,
            "page_size": 10, "page": page,
            "fields": "items.point,items.address,items.rubrics,items.reviews,items.rating",
            "key": api_key, "locale": "ru_RU",
        }
        resp = requests.get(BASE_URL, params=params, timeout=10)
        data = resp.json()
        meta_code = data.get("meta", {}).get("code")
        if meta_code != 200:
            break
        items = data.get("result", {}).get("items", [])
        if not items:
            break
        results.extend(items)
        total = data.get("result", {}).get("total", 0)
        if len(results) >= total or len(results) >= 200:
            break
        page += 1
        time.sleep(0.3)
    return results


def collect_competitors(client: ClientInput, api_key: str) -> pd.DataFrame:
    btype = BUSINESS_TYPE_MAP.get(client.business_type)
    if not btype:
        raise ValueError(f"неизвестный тип бизнеса: {client.business_type}")
    raw_items = []
    for query in btype["queries"]:
        items = fetch_2gis_places(query, client.lat, client.lon, client.search_radius_m, api_key)
        raw_items.extend(items)
    seen_ids = set()
    unique_items = []
    for item in raw_items:
        if item.get("id") not in seen_ids:
            seen_ids.add(item["id"])
            unique_items.append(parse_place(item))
    return pd.DataFrame(unique_items)


def haversine(lat1, lon1, lat2, lon2) -> float:
    R = 6371000
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2) ** 2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))


def category_match_score(row: pd.Series, btype_config: dict) -> float:
    text = (row["name"] + " " + row["rubrics"]).lower()
    direct_hits   = sum(1 for kw in btype_config["direct_keywords"]   if kw.lower() in text)
    indirect_hits = sum(1 for kw in btype_config["indirect_keywords"] if kw.lower() in text)
    if direct_hits > 0:
        return min(1.0, 0.6 + 0.1 * direct_hits)
    if indirect_hits > 0:
        return min(0.5, 0.2 + 0.1 * indirect_hits)
    return 0.1


def audience_overlap_score(client: ClientInput) -> float:
    if not client.target_audience:
        return 0.5
    btype = BUSINESS_TYPE_MAP[client.business_type]
    relevant_formats = set(btype.get("direct_keywords", []) + btype.get("indirect_keywords", []))
    overlap = 0
    for aud in client.target_audience:
        aud_config = TARGET_AUDIENCE_WEIGHTS.get(aud, {})
        pref = set(f.lower() for f in aud_config.get("preferred_formats", []))
        overlap += len(pref & relevant_formats)
    return min(1.0, 0.3 + overlap * 0.1)


def rating_score(rating: Optional[float], review_count: int) -> float:
    if rating is None:
        return 0.3
    review_weight = min(1.0, np.log1p(review_count) / np.log1p(500))
    return (rating / 5.0) * 0.6 + review_weight * 0.4


def proximity_score(distance_m: float, max_radius_m: int) -> float:
    return max(0.0, 1.0 - (distance_m / max_radius_m))


def compute_competition_coefficient(df: pd.DataFrame, client: ClientInput) -> pd.DataFrame:
    if df.empty:
        return df
    btype_config   = BUSINESS_TYPE_MAP[client.business_type]
    audience_weight = audience_overlap_score(client)
    df = df.copy()
    df["distance_m"] = df.apply(
        lambda r: haversine(client.lat, client.lon, r["lat"], r["lon"]) if pd.notna(r["lat"]) else np.nan,
        axis=1, result_type="reduce",
    )
    df["score_category"]  = df.apply(lambda r: category_match_score(r, btype_config), axis=1, result_type="reduce")
    df["score_rating"]    = df.apply(lambda r: rating_score(r["rating"], r["review_count"]), axis=1, result_type="reduce")
    df["score_proximity"] = df["distance_m"].apply(
        lambda d: proximity_score(d, client.search_radius_m) if pd.notna(d) else 0.0
    )
    df["competition_score"] = (
        df["score_category"]  * 0.45 +
        df["score_proximity"] * 0.30 +
        df["score_rating"]    * 0.25
    ) * audience_weight
    df["competition_score"] = df["competition_score"].clip(0, 1).round(3)

    def label(s):
        if s >= 0.65: return "Прямой"
        if s >= 0.40: return "Косвенный"
        return "Слабый"

    df["competition_label"] = df["competition_score"].apply(label)
    return df.sort_values("competition_score", ascending=False).reset_index(drop=True)


## 4. Анализ одного района (Хамовники)

In [17]:
client = ClientInput(
    business_type="coffeeshop",
    target_audience=["students", "office_workers"],
    location_name="Москва, район Хамовники",
    lat=55.7312,
    lon=37.5766,
    search_radius_m=SEARCH_RADIUS_M,
)

raw_df = collect_competitors(client, TWOGIS_API_KEY)
print(f"найдено объектов: {len(raw_df)}")

result_df = compute_competition_coefficient(raw_df, client)

cols = ["name", "rubrics", "address", "rating", "review_count", "distance_m",
        "competition_score", "competition_label"]
result_df[cols].head(20)


найдено объектов: 98


,name,rubrics,address,rating,review_count,distance_m,competition_score,competition_label
0,"Surf Coffee x East West, кофейня","Точки кофе, Точки безалкогольных напитков","улица Усачёва, 3",4.8,86,135.311071,0.248,Слабый
1,"Milk&Beans, кофейня",Кофейни,"улица Усачёва, 2 ст1",5.0,4,42.591370,0.232,Слабый
2,"Даблби, кофейня",Кофейни,"Оболенский переулок, 9 к1",4.5,19,191.886401,0.218,Слабый
3,"Cofix, кофейня","Точки кофе, Кондитерские изделия","переулок Хользунова, 6",4.8,42,292.267583,0.213,Слабый
4,"Skuratov Coffee, кофейня",Кофейни,"Комсомольский проспект, 24 ст1",4.6,74,456.064634,0.209,Слабый
5,"One Price Coffee, кофейня",Точки кофе,"переулок Хользунова, 1",4.0,27,397.033158,0.205,Слабый
6,"Abc Coffee Roasters, кофейня",Кофейни,"улица Усачёва, 11и",4.8,76,571.195537,0.198,Слабый
7,"Stars Coffee, кофейня",Кофейни,"Комсомольский проспект, 28",4.0,53,512.976675,0.196,Слабый
8,"Правда Кофе, экспресс-кофейня","Точки кофе, Продажа кофе","Комсомольский проспект, 28",4.9,35,488.463215,0.191,Слабый
9,"Yummy mummy, кафе","Кофейни, Доставка еды, Кулинарии","улица Еланского, 2 ст1",5.0,1,384.210591,0.190,Слабый


In [13]:
resp = requests.get(BASE_URL, params={
    "q": "кофейня",
    "point": f"{client.lon},{client.lat}",
    "radius": client.search_radius_m,
    "key": TWOGIS_API_KEY,
})
print(resp.status_code)
print(resp.json().get("meta"))
print(len(resp.json().get("result", {}).get("items", [])))

200
{'api_version': '3.0.20462', 'code': 403, 'error': {'message': 'Authorization error, incorrect key.', 'type': 'forbidden'}, 'issue_date': '20260526'}
0


In [18]:
# карта одного района
COLOR_MAP = {"Прямой": "red", "Косвенный": "orange", "Слабый": "green"}

m = folium.Map(location=[client.lat, client.lon], zoom_start=15)
folium.Circle(location=[client.lat, client.lon], radius=client.search_radius_m,
              color="blue", fill=True, fill_opacity=0.05).add_to(m)
folium.Marker(location=[client.lat, client.lon], popup="потенциальная точка",
              icon=folium.Icon(color="blue", icon="star")).add_to(m)

for _, row in result_df.dropna(subset=["lat", "lon"]).iterrows():
    color = COLOR_MAP.get(row["competition_label"], "gray")
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=6 + row["competition_score"] * 8,
        color=color, fill=True, fill_opacity=0.7,
        popup=folium.Popup(
            f"<b>{row['name']}</b><br>{row['address']}<br>"
            f"рейтинг: {row['rating']} ({row['review_count']} отзывов)<br>"
            f"расстояние: {row['distance_m']:.0f} м<br>"
            f"<b>скор: {row['competition_score']}</b>", max_width=280)
    ).add_to(m)
m


## 5. Сбор по всем районам

In [19]:
def collect_district_stats(district, okrug, lat, lon):
    # создаём ClientInput для каждого района и используем оригинальные функции
    district_client = ClientInput(
        business_type=BUSINESS_TYPE,
        target_audience=TARGET_AUDIENCE,
        location_name=district,
        lat=lat,
        lon=lon,
        search_radius_m=SEARCH_RADIUS_M,
    )
    raw = collect_competitors(district_client, TWOGIS_API_KEY)
    if raw.empty:
        return {"district": district, "okrug": okrug,
                "n_competitors": 0, "avg_comp_score": 0.0, "avg_comp_rating": np.nan}

    scored = compute_competition_coefficient(raw, district_client)
    return {
        "district":        district,
        "okrug":           okrug,
        "n_competitors":   len(scored),
        "avg_comp_score":  scored["competition_score"].mean().round(3),
        "avg_comp_rating": scored["rating"].dropna().mean().round(2) if scored["rating"].notna().any() else np.nan,
    }


rows = []
for district, okrug, lat, lon in tqdm(MOSCOW_DISTRICTS, desc="районы"):
    rows.append(collect_district_stats(district, okrug, lat, lon))

df_comp_all = pd.DataFrame(rows)
print(f"готово: {len(df_comp_all)} районов")
df_comp_all.head(10)


районы: 100%|██████████| 70/70 [08:13<00:00,  7.05s/it]

готово: 70 районов


,district,okrug,n_competitors,avg_comp_score,avg_comp_rating
0,Хамовники,ЦАО,93,0.142,4.39
1,Арбат,ЦАО,103,0.150,4.39
2,Тверской,ЦАО,104,0.148,4.55
3,Пресненский,ЦАО,97,0.145,4.45
4,Замоскворечье,ЦАО,102,0.142,4.52
5,Якиманка,ЦАО,100,0.141,4.25
6,Басманный,ЦАО,99,0.119,4.45
7,Таганский,ЦАО,104,0.157,4.35
8,Мещанский,ЦАО,100,0.154,4.51
9,Красносельский,ЦАО,107,0.137,4.22


## 6. Сохранение

In [20]:
import os
os.makedirs("darkspot_data", exist_ok=True)
df_comp_all.to_csv("darkspot_data/competitors.csv", index=False, encoding="utf-8-sig")
print(f"сохранено: darkspot_data/competitors.csv ({len(df_comp_all)} строк)")


сохранено: darkspot_data/competitors.csv (70 строк)
